In [1]:
import pandas as pd
import numpy as np
import faiss

from scipy.sparse import csr_matrix, save_npz
from implicit.als import AlternatingLeastSquares

from catboost import CatBoostRanker, Pool
from sklearn.model_selection import ParameterSampler

In [2]:
df = pd.read_parquet("../data/res/int.parquet")

df = df.sort_values("date").reset_index(drop=True)

In [3]:
embeddings = np.load(
    "content_embeddings.npy",
    mmap_mode="r"
)

item_index = pd.read_parquet(
    "content_item_index.parquet"
)

content_index = faiss.read_index(
    "content_faiss.index",
    faiss.IO_FLAG_MMAP
)

content_index.nprobe = 50

In [4]:
asin_to_idx = item_index.set_index("parent_asin")["item_idx"]
idx_to_asin = item_index["parent_asin"].to_numpy()

In [5]:
def hit_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    return int(len(set(recom) & true_ans) > 0)

def recall_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    hits = len(set(recom) & true_ans)

    return hits / len(true_ans)

def ndcg_at_k(recom, true_ans, k):
    recom = recom[:k]
    true_ans = set(true_ans)

    if len(true_ans) == 0:
        return 0

    dcg = 0

    for rank, item in enumerate(recom):
        if item in true_ans:
            dcg += 1 / np.log2(rank + 2)

    ideal_hits = min(len(true_ans), k)

    idcg = sum(
        1 / np.log2(rank + 2)
        for rank in range(ideal_hits)
    )

    return dcg / idcg

In [6]:
def make_targets(df, sepor, target_end):
    history = df[df["date"] < sepor]

    future = df[
        (df["date"] >= sepor) &
        (df["date"] < target_end) &
        (df["rating"] >= 4)
    ].copy()

    future = future[future["user_id"].isin(history["user_id"])]

    seen = history[["user_id", "parent_asin"]].drop_duplicates()

    future = future.merge(
        seen.assign(seen=1),
        on=["user_id", "parent_asin"],
        how="left"
    )

    future = future[future["seen"].isna()]

    return future.groupby("user_id")["parent_asin"].agg(set)

In [7]:
def fit_generators(df, sepor, days=90):
    start = sepor - pd.Timedelta(days=days)

    history = df[
        (df["date"] >= start) &
        (df["date"] < sepor)
    ].copy()

    user_seen = df[df['date'] < sepor].groupby("user_id")["parent_asin"].agg(set)

    positive = history[history["rating"] >= 4]

    pop_counts = positive.groupby("parent_asin").size().sort_values(ascending=False)
    popular_items = pop_counts.index.tolist()
    
    als_data = history.sort_values("date").drop_duplicates(
            ["user_id", "parent_asin"],
            keep="last"
        ).copy()
    

    
    als_data = als_data[als_data["rating"] >= 4].copy()
    als_data["weight"] = 1.0
    

    user_codes, user_ids = pd.factorize(als_data["user_id"])
    item_codes, item_ids = pd.factorize(als_data["parent_asin"])

    user_item = csr_matrix(
        (als_data["weight"].values,
        (user_codes, item_codes)),
        shape=(len(user_ids), len(item_ids)))

    als_model = AlternatingLeastSquares(
        factors=64,
        regularization=0.05,
        alpha=20,
        iterations=30,
        random_state=777
    )

    als_model.fit(user_item, show_progress=False)

    user_to_idx = {
        user_id: idx
        for idx, user_id in enumerate(user_ids)
    }

    content_data = history.sort_values("date").drop_duplicates(["user_id", "parent_asin"],keep="last")
    content_data = content_data[content_data["rating"] >= 4].copy()
    content_data = content_data[content_data["parent_asin"].isin(asin_to_idx.index)]
    content_data["item_idx"] = content_data["parent_asin"].map(asin_to_idx).astype(int)
   

    content_history = content_data.groupby("user_id")["item_idx"].agg(list)
    
    return {
        "popular_items": popular_items,
        "user_seen": user_seen,
        "pop_counts": pop_counts.to_dict(),
        
        "als_model": als_model,
        "als_user_item": user_item,
        "als_user_ids": np.asarray(user_ids),
        "als_item_ids": np.asarray(item_ids),
        "als_user_to_idx": user_to_idx,

        "content_history": content_history
    }

In [8]:
def get_items(recs):
    if len(recs) == 0:
        return []

    if isinstance(recs[0], tuple):
        return [item for item, score in recs]

    return recs

In [9]:
def recommend_popular(user_id, generators, k):
    seen = generators["user_seen"].get(user_id, set())

    recs = []

    for item in generators["popular_items"]:
        if item not in seen:
            recs.append(item)

        if len(recs) == k:
            break

    return recs


In [10]:
def recommend_als(user_id, generators, k):
    user_idx = generators["als_user_to_idx"].get(user_id)

    if user_idx is None:
        return []

    model = generators["als_model"]
    user_item = generators["als_user_item"]
    item_ids = generators["als_item_ids"]

    search_k = min(k + 500, len(item_ids))

    item_idx, scores = model.recommend(
        userid=user_idx,
        user_items=user_item[user_idx],
        N=search_k,
        filter_already_liked_items=True
    )

    seen = generators["user_seen"].get(user_id, set())

    recs = []

    for idx, score in zip(item_idx, scores):
        item = item_ids[idx]

        if item not in seen:
            recs.append((item, score))

        if len(recs) == k:
            break

    return recs

In [11]:
def recommend_content(user_id, generators, k, neighbors_per_item=1000):
    history = generators["content_history"]

    if user_id not in history.index:
        return []

    history_idx = history[user_id]
    queries = np.asarray(embeddings[history_idx], dtype=np.float32)

    scores, neighbors = content_index.search(queries, neighbors_per_item)
    seen = generators["user_seen"].get(user_id, set())

    candidates = {}

    for idx, score in zip(neighbors.ravel(), scores.ravel()):

        if idx == -1:
            continue

        item = idx_to_asin[idx]

        if item in seen:
            continue

        if item not in candidates or score > candidates[item]:
            candidates[item] = score

    ranked = sorted(
        candidates.items(),
        key=lambda x: x[1],
        reverse=True
        )

    return ranked[:k]

In [12]:
def recommend_user(user_id, generators, k=1000):
    
    return {
        "popularity": recommend_popular(user_id, generators, k),
        "als": recommend_als(user_id, generators, k),
        "content": recommend_content(user_id, generators, k)
    }

In [13]:
def generate_candidates(users, generators, k=1000):
    recommendations = {
        "popularity": {},
        "als": {},
        "content": {}
    }

    for user_id in users:
        user_recs = recommend_user(
            user_id,
            generators,
            k
        )

        for name in recommendations:
            recommendations[name][user_id] = user_recs[name]

    return recommendations

In [ ]:
def prepare_period(df, sepor, target_end, days=90, k=1000):
    targets = make_targets(df, sepor, target_end)

    generators = fit_generators(df, sepor, days=days)

    recommendations = generate_candidates(targets.index, generators, k=k)
    pop_counts = generators["pop_counts"]


    return {
        "targets": targets,
        "generators": {"pop_counts": pop_counts},
        "recommendations": recommendations
    }

### Схема оценки генераторов

Для каждого временного среза сначала формируется история до даты отсечения, а таргетом считаются только положительные взаимодействия (rating >= 4) из следующего месяца с товарами, которых пользователь раньше не оценивал. Генераторы обучаются только на прошлом относительно этого среза

На этом этапе k=1000 используется не как финальный размер выдачи, а как размер пула кандидатов: задача генераторов обеспечить как можно больший охват будущих релевантных товаров, а окончательный порядок внутри объединённого пула будет определять ранжировщик.

In [15]:
def compare_generators(recommendations, targets, k, ranking=False):
    rows = []

    for name in ["popularity", "als", "content"]:
        hits = []
        recalls = []
        ndcgs = []

        for user_id, true_items in targets.items():
            recs = get_items(recommendations[name].get(user_id, []))

            if name == "als" and len(recs) == 0:
                continue
            
            hits.append(hit_at_k(recs, true_items, k))

            recalls.append(recall_at_k(recs, true_items, k))

            if ranking:
                ndcgs.append(ndcg_at_k(recs, true_items, k))

        row = {
            "generator": name,
            "hit_rate": np.mean(hits),
            "recall": np.mean(recalls)
        }

        if ranking:
            row["ndcg"] = np.mean(ndcgs)

        rows.append(row)

    return pd.DataFrame(rows)

In [16]:
VAL_SEPOR = pd.Timestamp("2023-07-14")
VAL_END = pd.Timestamp("2023-08-14")

val_period = prepare_period(
    df,
    sepor=VAL_SEPOR,
    target_end=VAL_END,
    days=90,
    k=1000)

/home/user/mle/venv/lib/python3.14/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [ ]:
results_1000 = compare_generators(
    val_period["recommendations"],
    val_period["targets"],
    k=1000
)

results_1000

,generator,hit_rate,recall
0,popularity,0.180022,0.161246
1,als,0.161636,0.129004
2,content,0.024458,0.020383


In [ ]:
results_10 = compare_generators(
    val_period["recommendations"],
    val_period["targets"],
    k=10,
    ranking=True
)

results_10

,generator,hit_rate,recall,ndcg
0,popularity,0.008373,0.007779,0.003723
1,als,0.025942,0.020353,0.013484
2,content,0.007124,0.005886,0.004393


На глубине 1000 popularity даёт самый широкий охват кандидатов, тогда как на первых 10 позициях ALS показывает более сильный персонализированный порядок. Content-модель заметно слабее как самостоятельный генератор, но это ещё не означает, что её нужно исключать: для ранжирования важна не только собственная метрика генератора, но и наличие релевантных кандидатов, которых не находят другие методы.


In [16]:
def unique_correct_answers(recommendations, targets, k=1000):
    names = ["popularity", "als", "content"]

    total_correct = {name: 0 for name in names}

    unique_correct = {name: 0 for name in names}

    users_with_unique = {name: 0 for name in names}

    for user_id, true_items in targets.items():
        hits = {}

        for name in names:
            recs = get_items(recommendations[name].get(user_id, []))[:k]

            hits[name] = (set(recs) & set(true_items))

        for name in names:
            other_hits = set()

            for other in names:
                if other != name:
                    other_hits.update(hits[other])

            unique = hits[name] - other_hits

            total_correct[name] += len(hits[name])
            unique_correct[name] += len(unique)

            if len(unique) > 0:
                users_with_unique[name] += 1

    rows = []

    for name in names:
        rows.append({
            "generator": name,
            "correct_targets": total_correct[name],
            "unique_correct_targets": unique_correct[name],
            "users_with_unique_hit": users_with_unique[name]
        })

    return pd.DataFrame(rows)

In [ ]:
unique_results = unique_correct_answers(
    val_period["recommendations"],
    val_period["targets"],
    k=1000
)

unique_results

,generator,correct_targets,unique_correct_targets,users_with_unique_hit
0,popularity,2536,2190,2130
1,als,738,356,309
2,content,369,279,250


Наибольшее число правильных и уникальных попаданий даёт popularity-модель. При этом ALS и content также находят релевантные товары, которых нет в рекомендациях других генераторов. Особенно важно, что из 369 правильных товаров, найденных content-моделью, 279 являются уникальными.

Таким образом, даже генератор с более низкими индивидуальными метриками может быть полезен при формировании общего пула кандидатов, поскольку добавляет релевантные товары, которые не были найдены другими методами.

In [17]:
def compare_combinations(recommendations, targets):
    combinations = [
        ("popularity",),
        ("als",),
        ("content",),
        ("popularity", "als"),
        ("popularity", "content"),
        ("als", "content"),
        ("popularity", "als", "content")
    ]

    rows = []

    for combination in combinations:
        hits = []
        recalls = []
        sizes = []

        for user_id, true_items in targets.items():
            candidates = set()

            for name in combination:
                candidates.update(get_items(recommendations[name].get(user_id, [])))
                
            hits.append(hit_at_k(list(candidates), true_items, len(candidates)))
            recalls.append(recall_at_k(list(candidates), true_items, len(candidates)))
            sizes.append(len(candidates))

        rows.append({
            "generators": " + ".join(combination),
            "avg_candidates": np.mean(sizes),
            "hit_rate": np.mean(hits),
            "recall": np.mean(recalls)
        })

    return pd.DataFrame(rows)

In [ ]:
combination_results = compare_combinations(val_period["recommendations"], val_period["targets"])
combination_results

,generators,avg_candidates,hit_rate,recall
0,popularity,1000.000000,0.180022,0.161246
1,als,294.454646,0.047595,0.037986
2,content,294.264194,0.024458,0.020383
3,popularity + als,1193.832391,0.203085,0.179857
4,popularity + content,1293.161219,0.200073,0.178716
5,als + content,587.407712,0.065222,0.053512
6,popularity + als + content,1486.209989,0.219905,0.194869


Каждый генератор находит часть уникальных правильных ответов. На validation объединение popularity + ALS + content повышает HitRate пула кандидатов до 0.2199, а Recall до 0.1949, что выше любого отдельного источника.

Поэтому в ранжировщик передаётся объединение всех трёх списков. На этом этапе важнее сохранить разнообразие и полноту кандидатов, чем получить хороший порядок: ранжировщик будет решать, какие из найденных объектов поднять в top-10.

In [18]:
RANK_sepor = pd.Timestamp("2023-06-14")
RANK_END = pd.Timestamp("2023-07-14")

TEST_sepor = pd.Timestamp("2023-08-14")
TEST_END = pd.Timestamp("2023-09-14")

Ранжировщик обучается на более раннем месяце: история заканчивается 2023-06-14, а положительные таргеты берутся из интервала 2023-06-14 — 2023-07-14. Следующий месяц (2023-07-14 — 2023-08-14) используется для выбора гиперпараметров, а период 2023-08-14 — 2023-09-14 оставлен для финальной проверки.

Для каждого периода генераторы строятся заново только по данным до соответствующей даты отсечения. Такое временное разделение важно для рекомендательной системы, поскольку не позволяет использовать информацию из будущего при построении кандидатов.

In [20]:
rank_period = prepare_period(
    df,
    sepor=RANK_sepor,
    target_end=RANK_END,
    days=90,
    k=1000)

In [19]:
RANK_FEATURES = [
    "pop_rank",
    "als_rank",
    "content_rank",

    "from_pop",
    "from_als",
    "from_content",

    "source_count",
    "best_rank",

    "pop_score",
    "als_score",
    "content_score"
]

### Признаки ранжирования

Признаки описывают не сам товар, а то, как три генератора оценивают одного и того же кандидата:

- `pop_rank`, `als_rank`, `content_rank` — позиция кандидата внутри каждого генератора; отсутствие в списке кодируется рангом 1001;
- `from_pop`, `from_als`, `from_content` — бинарные признаки источника;
- `source_count` — число генераторов, которые независимо предложили товар, то есть простой сигнал согласия моделей;
- `best_rank` — лучшая позиция кандидата среди всех источников;
- `pop_score`, `als_score`, `content_score` — исходная сила сигнала генератора: частота положительных взаимодействий, ALS-score и близость в пространстве контентных эмбеддингов.

Такой набор позволяет CatBoost учиться комбинировать сигналы генераторов. При этом пользовательские и товарные характеристики здесь отсутствуют, поэтому персонализация ранжировщика ограничена той информацией, которая уже содержится в ALS/content-выдаче и их позициях.

In [20]:
def make_rank_rows(user_id, recommendations, generators, targets=None, n_negatives=None):
    pop = recommendations["popularity"].get(user_id, [])
    als = recommendations["als"].get(user_id, [])
    content = recommendations["content"].get(user_id, [])
    pop_counts = generators["pop_counts"]

    pop_rank = {
        item: rank
        for rank, item in enumerate(pop, start=1)
        }
    als_rank = {
        item: rank
        for rank, (item, score) in enumerate(als, start=1)
        }

    als_score = {
        item: score
        for item, score in als
        }

    content_rank = {
        item: rank
        for rank, (item, score) in enumerate(content, start=1)
        }

    content_score = {
        item: score
        for item, score in content
        }

    candidates = (set(pop_rank) | set(als_rank) | set(content_rank))

    if targets is not None:
        targets = set(targets)

        positive = candidates & targets

        if len(positive) == 0:
            return []

        if n_negatives is not None:
            negatives = candidates - targets

            def best_candidate_rank(item):
                return min(
                    pop_rank.get(item, 1001),
                    als_rank.get(item, 1001),
                    content_rank.get(item, 1001)
                )

            negatives = sorted(negatives, key=best_candidate_rank)[:n_negatives]

            candidates = positive | set(negatives)

    rows = []

    for item in candidates:
        pr = pop_rank.get(item, 1001)
        ar = als_rank.get(item, 1001)
        cr = content_rank.get(item, 1001)
        

        fp = int(item in pop_rank)
        fa = int(item in als_rank)
        fc = int(item in content_rank)

        row = {
            "user_id": user_id,
            "parent_asin": item,

            "pop_rank": pr,
            "als_rank": ar,
            "content_rank": cr,

            "from_pop": fp,
            "from_als": fa,
            "from_content": fc,
            "pop_score": pop_counts.get(item, 0),
            "als_score": als_score.get(item, 0),
            "content_score": content_score.get(item, 0),
            "source_count": fp + fa + fc,
            "best_rank": min(pr, ar, cr)
        }

        if targets is not None:
            row["label"] = int(item in targets)

        rows.append(row)

    return rows

In [21]:
def make_rank_dataset(period, period_name, n_negatives=500):
    rows = []

    for user_id, targets in period["targets"].items():
        user_rows = make_rank_rows(
            user_id,
            period["recommendations"],
            period["generators"],
            targets=targets,
            n_negatives=n_negatives
        )

        for row in user_rows:
            row["group_key"] = (
                period_name
                + "_"
                + str(user_id)
            )

        rows.extend(user_rows)

    return pd.DataFrame(rows)

In [24]:
rank_train = make_rank_dataset(
    rank_period,
    period_name="train",
    n_negatives=500
)

print("Rows:", len(rank_train))
print("Groups:", rank_train["group_key"].nunique())

rank_train["label"].value_counts()

Rows: 2094952
Groups: 4181


label
0    2090500
1       4452
Name: count, dtype: int64

Положительным классом являются кандидаты, которые действительно встретились у пользователя в следующем временном окне. Для каждого пользователя сохраняются все найденные положительные кандидаты и до 500 отрицательных. Отрицательные выбираются не случайно, а среди кандидатов с лучшими позициями в генераторах — это «трудные» отрицательные примеры, на которых полезнее учиться различать верх выдачи.

После фильтрации получилось 4 452 положительных объекта против 2 090 500 отрицательных. Кроме того, в train попадают только группы, где хотя бы один будущий положительный товар уже присутствует в кандидатном пуле: если генераторы не нашли ни одного таргета, ранжировщик физически не может его восстановить.


In [22]:
def make_pool(data):
    data = data.sort_values("group_key").reset_index(drop=True)
    group_id = pd.factorize(data["group_key"])[0]

    pool = Pool(
        data=data[RANK_FEATURES],
        label=data["label"],
        group_id=group_id
    )

    return data, pool

In [26]:
rank_train, train_pool = make_pool(rank_train)


In [23]:
def rank_candidates(user_id, recommendations, generators, ranker, k=10):
    rows = make_rank_rows(user_id, recommendations, generators)

    if len(rows) == 0:
        return []

    data = pd.DataFrame(rows)

    data["score"] = ranker.predict(data[RANK_FEATURES])

    return data.sort_values("score", ascending=False)["parent_asin"].head(k).tolist()
    
    

In [24]:
def score_ranker(model, period, k=10):
    hits = []
    recalls = []
    ndcgs = []

    for user_id, targets in period["targets"].items():
        recs = rank_candidates(user_id, period["recommendations"], period["generators"], model, k=k)

        hits.append(hit_at_k(recs, targets, k))
        recalls.append(recall_at_k(recs, targets, k))
        ndcgs.append(ndcg_at_k(recs, targets, k))

    return {
        "hit_rate": np.mean(hits),
        "recall": np.mean(recalls),
        "ndcg": np.mean(ndcgs)
    }

In [30]:
param_grid = {
    "iterations": [400, 600, 800],
    "learning_rate": [0.03, 0.05, 0.08],
    "depth": [5, 6, 7],
    "l2_leaf_reg": [3, 5, 8]
}

param_list = list(
    ParameterSampler(
        param_grid,
        n_iter=10,
        random_state=777
    )
)

In [33]:
search_results = []

best_score = -1
best_params = None
best_model = None

for params in param_list:
    model = CatBoostRanker(
        loss_function="YetiRankPairwise",
        iterations=params["iterations"],
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],
        random_seed=777,
        task_type="GPU",
        devices="0",
        verbose=False
    )

    model.fit(train_pool)

    metrics = score_ranker(
        model,
        val_period,
        k=10
    )

    score = metrics["ndcg"]

    search_results.append({
        "iterations": params["iterations"],
        "learning_rate": params["learning_rate"],
        "depth": params["depth"],
        "l2_leaf_reg": params["l2_leaf_reg"],
        "hit_rate": metrics["hit_rate"],
        "recall": metrics["recall"],
        "ndcg_at_10": score
    })

    if score > best_score:
        best_score = score
        best_params = params.copy()
        best_model = model

    print(
        params,
        "NDCG@10:",
        round(score, 5)
    )

Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


{'learning_rate': 0.03, 'l2_leaf_reg': 8, 'iterations': 400, 'depth': 7} NDCG@10: 0.00032


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


{'learning_rate': 0.05, 'l2_leaf_reg': 3, 'iterations': 400, 'depth': 5} NDCG@10: 0.0003


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


...

{'learning_rate': 0.05, 'l2_leaf_reg': 3, 'iterations': 800, 'depth': 5} NDCG@10: 0.00029


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


{'learning_rate': 0.05, 'l2_leaf_reg': 8, 'iterations': 800, 'depth': 6} NDCG@10: 0.00029


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


{'learning_rate': 0.03, 'l2_leaf_reg': 8, 'iterations': 600, 'depth': 7} NDCG@10: 0.00026


In [34]:
search_results = pd.DataFrame(search_results).sort_values("ndcg_at_10", ascending=False)
search_results

,iterations,learning_rate,depth,l2_leaf_reg,hit_rate,recall,ndcg_at_10
3,800,0.05,7,3,0.000881,0.000670,0.000367
6,800,0.05,5,8,0.001102,0.000888,0.000352
0,400,0.03,7,8,0.000661,0.000588,0.000315
1,400,0.05,5,3,0.001102,0.000869,0.000301
2,600,0.08,5,3,0.001102,0.000788,0.000299
7,800,0.05,5,3,0.001028,0.000832,0.000290
8,800,0.05,6,8,0.000955,0.000741,0.000285
9,600,0.03,7,8,0.000881,0.000722,0.000261
4,800,0.05,5,5,0.000881,0.000722,0.000258
5,600,0.08,6,8,0.000881,0.000686,0.000245


In [35]:
best_params

{'learning_rate': 0.05, 'l2_leaf_reg': 3, 'iterations': 800, 'depth': 7}

Гиперпараметры перебираются случайной выборкой из заданной сетки, а лучшая конфигурация выбирается по NDCG@10. Для финальной задачи это важнее Recall@1000: после генерации кандидатов нас уже интересует не наличие таргета где-то в большом пуле, а его положение в первых десяти рекомендациях.

Лучшей на validation стала конфигурация `iterations=800`, `learning_rate=0.05`, `depth=7`, `l2_leaf_reg=3`. Однако абсолютное значение NDCG@10 ≈ 0.00037 очень низкое. Поэтому преимущество этой конфигурации над остальными стоит трактовать только как выбор лучшего варианта внутри текущей ситуации, а не как признак хорошего качества ранжировщика. 

In [36]:
ranker = best_model

In [37]:
ranker.save_model("catboost_ranker.cbm")

In [25]:
ranker = CatBoostRanker()
ranker.load_model('catboost_ranker.cbm')

CatBoostRanker(depth=7, devices='0', iterations=800, l2_leaf_reg=3, learning_rate=0.05, loss_function='YetiRankPairwise', random_seed=777, task_type='GPU', verbose=0)

In [26]:
test_period = prepare_period(
    df,
    sepor=TEST_sepor,
    target_end=TEST_END,
    days=90,
    k=1000)

/home/user/mle/venv/lib/python3.14/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


In [27]:
test_quality = compare_generators(
    test_period["recommendations"],
    test_period["targets"],
    k=10,
    ranking=True
)

In [28]:
ranker_metrics = score_ranker(ranker, test_period, k=10)
ranker_metrics

{'hit_rate': np.float64(0.001461017399389029),
 'recall': np.float64(0.0011001903749944658),
 'ndcg': np.float64(0.0006708577380616095)}

In [29]:
test_quality = compare_generators(
    test_period["recommendations"],
    test_period["targets"],
    k=10,
    ranking=True
)

ranker_row = pd.DataFrame([{
    "generator": "catboost_ranker",
    "hit_rate": ranker_metrics["hit_rate"],
    "recall": ranker_metrics["recall"],
    "ndcg": ranker_metrics["ndcg"]
}])

test_quality = pd.concat([test_quality, ranker_row], ignore_index=True)

test_quality

,generator,hit_rate,recall,ndcg
0,popularity,0.009696,0.009320,0.004155
1,als,0.029986,0.024326,0.013467
2,content,0.009430,0.007226,0.005063
3,catboost_ranker,0.001461,0.001100,0.000671


In [30]:
test_combinations = compare_combinations(
    test_period["recommendations"],
    test_period["targets"]
)

test_combinations

,generators,avg_candidates,hit_rate,recall
0,popularity,1000.000000,0.158985,0.141932
1,als,367.645106,0.069730,0.055955
2,content,367.427547,0.032408,0.026240
3,popularity + als,1236.026298,0.197370,0.172417
4,popularity + content,1366.212910,0.185948,0.164581
5,als + content,733.550272,0.094568,0.076916
6,popularity + als + content,1601.304821,0.219950,0.191673


### Итоговая оценка

На тестовом периоде CatBoostRanker получил HitRate@10 = 0.00146, Recall@10 = 0.00110 и NDCG@10 = 0.00067. Это заметно хуже popularity и content на той же общей выборке. 

При этом объединённый пул трёх генераторов на тесте имеет HitRate ≈ 0.2200 и Recall ≈ 0.1917. Следовательно, основное ограничение текущей системы находится уже не в генерации кандидатов: релевантные товары достаточно часто присутствуют в пуле, но ранжировщик почти не поднимает их в top-10.

Вероятная причина — слишком ограниченный набор признаков: модель в основном учится глобально переоценивать позиции и scores трёх генераторов, но почти не получает отдельной информации о пользователе, товаре и контексте взаимодействия. 